In [ ]:
%load_ext autoreload
%autoreload 2

# Pefrom Wiener-SVD Unfolding

In [ ]:
from os import path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append('../../../')
from analysis_village.unfolding.utils import *

from var_config_cohpi import *


## 1. Open Covariance Files

In [ ]:
genie_cov = np.load("genie_syst_cov_matrices.npz", allow_pickle=True)
flux_cov = np.load("flux_syst_cov_matrices.npz", allow_pickle=True)
stat_cov = np.load("mcstat_syst_cov_matrices.npz", allow_pickle=True)

## 2. Asimov data test

### 2 - 1: Make Asimov data

In [ ]:
## for asimov data stat unc. calculation
XSEC_UNIT = 7.132e-43

asimov_data_sr = genie_cov['bs'] + genie_cov['ms']
asimov_data_sb = genie_cov['bc'] + genie_cov['mc']
## stat. unc of asimov data in sqrt(N) approximation, converted to xsec unit
cov_asimov_data_sr = np.diag(asimov_data_sr * XSEC_UNIT)
cov_asimov_data_sb = np.diag(asimov_data_sb * XSEC_UNIT)

### 2 - 2: Plot Asimov data vs. MC plot

In [ ]:
plot_overlay_with_cov(
    data=asimov_data_sr,
    cov_data=cov_asimov_data_sr,
    signal=genie_cov["ms"],
    cov_signal=genie_cov["cov_ms_ms"].item()["cov"] + flux_cov["cov_ms_ms"].item()["cov"] + stat_cov["cov_ms_ms"].item()["cov"],
    background=genie_cov["bs"],
    cov_background=genie_cov["cov_bs_bs"].item()["cov"] + flux_cov["cov_bs_bs"].item()["cov"] + stat_cov["cov_bs_bs"].item()["cov"],
    varcfg=varcfg_p_mu,
    title=f"SR: Asimov vs Prediction (flux, GENIE, MC stat. unc.)",
    data_label="Asimov Data (stat unc.)",
    y_label_top=r"Flux avg. $\sigma$ [cm$^2/\rm{Ar}$]",
    ratio_y_min=0.5,
    ratio_y_max=1.5,
)


In [ ]:
plot_overlay_with_cov(
    data=asimov_data_sr,
    cov_data=cov_asimov_data_sr,
    signal=genie_cov["ms"],
    cov_signal=genie_cov["cov_ms_ms"].item()["cov"],
    background=genie_cov["bs"],
    cov_background=genie_cov["cov_bs_bs"].item()["cov"],
    varcfg=varcfg_p_mu,
    title=f"SR: Asimov vs Prediction (GENIE syst.)",
    data_label="Asimov Data (stat unc.)",
    y_label_top=r"Flux avg. $\sigma$ [cm$^2/\rm{Ar}$]",
    ratio_y_min=0.5,
    ratio_y_max=1.5,
)


In [ ]:
plot_overlay_with_cov(
    data=asimov_data_sb,
    cov_data=cov_asimov_data_sb,
    signal=genie_cov["mc"],
    cov_signal=genie_cov["cov_mc_mc"].item()["cov"] + flux_cov["cov_mc_mc"].item()["cov"] + stat_cov["cov_mc_mc"].item()["cov"],
    background=genie_cov["bc"],
    cov_background=genie_cov["cov_bc_bc"].item()["cov"] + flux_cov["cov_bc_bc"].item()["cov"] + stat_cov["cov_bc_bc"].item()["cov"],
    varcfg=varcfg_p_mu,
    title=f"SB: Asimov vs Prediction (flux, GENIE, MC stat. unc.)",
    data_label="Asimov Data (stat unc.)",
    y_label_top=r"Flux avg. $\sigma$ [cm$^2/\rm{Ar}$]",
    ratio_y_min=0.5,
    ratio_y_max=1.5,
)


### 2 - 3: Event Rate Unc. Budget

In [ ]:
def plot_combined_uncertainty_budget(var_cfg, signal, bkg, sig_cov_dict, bkg_cov_dict, title="Uncertainty Budget"):
    """
    Plots fractional uncertainty per category (Sig + Bkg combined).
    Uses a distinct color cycle for categories and reserves Black for the Total.
    """
    bins = var_cfg.bins
    total_prediction = signal + bkg
    
    # Identify unique categories
    categories = sorted(set(sig_cov_dict.keys()) | set(bkg_cov_dict.keys()))
    
    # Use a standard qualitative color map (excludes black)
    colors = plt.cm.tab10.colors 
    
    plt.figure(figsize=(9, 6))
    total_cov = np.zeros((len(total_prediction), len(total_prediction)))

    for i, cat in enumerate(categories):
        s_cov = sig_cov_dict.get(cat, np.zeros_like(total_cov))
        b_cov = bkg_cov_dict.get(cat, np.zeros_like(total_cov))
        
        # Combine Sig and Bkg for this specific source
        cat_cov = s_cov + b_cov
        total_cov += cat_cov
        
        # Calculate fractional uncertainty
        unc = np.sqrt(np.diag(cat_cov))
        
        # Skip plotting if the category is empty/zero to save legend space
        if np.all(unc == 0): continue
            
        frac_unc = np.divide(unc, total_prediction, 
                             out=np.zeros_like(unc), 
                             where=total_prediction > 0)
        
        # Cycle through tab10 colors
        color = colors[i % len(colors)]
        
        plt.step(bins, np.append(frac_unc, frac_unc[-1]), where='post', 
                 label=cat, lw=1.5, color=color)

    # Calculate Grand Total
    total_unc_diag = np.sqrt(np.diag(total_cov))
    total_frac = np.divide(total_unc_diag, total_prediction, 
                           out=np.zeros_like(total_unc_diag), 
                           where=total_prediction > 0)
    
    # Explicitly use Black only for the Total
    plt.step(bins, np.append(total_frac, total_frac[-1]), where='post', 
             label='Total Uncertainty', color='black', linestyle='--', lw=2.5)

    # Formatting
    plt.xlabel(var_cfg.var_labels[0], fontsize=12)
    plt.ylabel(r"Fractional Uncertainty $\sigma / (N_{sig} + N_{bkg})$", fontsize=12)
    plt.title(f"{title}: {var_cfg.var_plot_name}", fontsize=14)
    plt.xlim(bins[0], bins[-1])
    plt.ylim(0, max(total_frac) * 1.4 if len(total_frac) > 0 else 0.2)
    
    # Move legend outside to prevent overlapping lines
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
    plt.grid(alpha=0.2, linestyle='--')
    
    plt.tight_layout()
    plt.show()

In [ ]:
sr_sig_covs_dict = {
    "GENIE": genie_cov["cov_ms_ms"].item()["cov"],
    "Flux": flux_cov["cov_ms_ms"].item()["cov"],
    "MC Stat.": stat_cov["cov_ms_ms"].item()["cov"],
}
sr_bkg_covs_dict = {
    "GENIE": genie_cov["cov_bs_bs"].item()["cov"],
    "Flux": flux_cov["cov_bs_bs"].item()["cov"],
    "MC Stat.": stat_cov["cov_bs_bs"].item()["cov"],
}

plot_combined_uncertainty_budget(varcfg_p_mu, genie_cov["ms"], genie_cov["bs"], sr_sig_covs_dict, sr_bkg_covs_dict, title="SR Uncertainty Budget")

### 2 - 4: Unfolding

In [ ]:
response = genie_cov["response"]
model = genie_cov["true_signal"] * XSEC_UNIT## true signal
measured = genie_cov["ms"] ## asimov - bkg -> true signal
#cov_total = stat_cov["cov_ms_ms"].item()["cov"]
cov_total = stat_cov["cov_ms_ms"].item()["cov"] + stat_cov["cov_bs_bs"].item()["cov"] + flux_cov["cov_ms_ms"].item()["cov"] + flux_cov["cov_bs_bs"].item()["cov"] # + genie_cov["cov_ms_ms"].item()["cov"] + genie_cov["cov_bs_bs"].item()["cov"]
C_type = 2
norm_type = 0
unfold = WienerSVD(response, model, measured, cov_total, C_type, norm_type)

In [ ]:
from scipy.stats import chi2

def get_chi2_pval(data, model, cov):
    this_chi2 = (data - model) @ np.linalg.inv(cov) @ (data - model)
    ndf = len(data)
    pval = 1 - chi2.cdf(this_chi2, ndf)
    return this_chi2, ndf, pval

def plot_unfolded_data_vs_truth(var_cfg, unfold_results, model_pred, reco_signal=None, title="Unfolded Result", get_chi2_func=None):
    """
    Plots unfolded data as points with error bars and the truth model as a step line.
    Includes chi2/ndf annotation.
    """
    # Extract unfolding data
    unfold_val = unfold_results['unfold']
    unfold_cov = unfold_results['UnfoldCov']
    unfold_err = np.sqrt(np.diag(unfold_cov))
    model_smear = unfold_results['AddSmear'] @ model_pred
    
    # Binning
    bins = var_cfg.bins
    bin_centers = (bins[:-1] + bins[1:]) / 2

    plt.figure(figsize=(8, 6))

    # 1. Unfolded Data: Points with Error Bars
    data_handle = plt.errorbar(
        bin_centers, unfold_val, yerr=unfold_err,
        fmt='ko', capsize=2, elinewidth=1.5, markersize=4,
        label='Unfolded Data'
    )

    # 2. Ac x True Signal: Step Line
    model_handle, = plt.step(
        bins, np.append(model_smear, model_smear[-1]),
        where='post', color='red', lw=2,
        label=r'$A_c \times$ True signal'
    )

    # 3. Optional: Reco Signal
    handles = [data_handle, model_handle]
    labels = ['Unfolded', r'$A_c \times$ True signal']

    if reco_signal is not None:
        reco_handle, = plt.plot(bin_centers, reco_signal, 'o', 
                                color='tab:blue', alpha=0.6, markersize=5, 
                                label='Reco. signal')
        handles.append(reco_handle)
        labels.append('Reco. signal')

    # --- Chi2 Calculation and Labeling ---
    chi2_val, ndf, pval = get_chi2_pval(unfold_val, model_smear, unfold_cov)
    chi2_text = r"$\chi^2/\rm{ndf} = %.2f / %d ~(p-value = %.2f)$" % (chi2_val, ndf, pval)
        
    # Placing the text in the upper right (axes coordinates: 0.95 is right, 0.90 is top)
    plt.text(0.15, 0.67, chi2_text, transform=plt.gca().transAxes, 
                fontsize=15, verticalalignment='top', horizontalalignment='left',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.5, edgecolor='none'))

    # Formatting
    plt.xlim(bins[0], bins[-1])
    plt.xlabel(var_cfg.var_labels[0])
    plt.ylabel(r"Flux avg. $\sigma$ [cm$^2/\rm{Ar}$]")
    plt.title(title)
    plt.legend(handles, labels, frameon=False, loc='upper left')
    plt.grid(alpha=0.2, linestyle='--')
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_unfolded_data_vs_truth(varcfg_p_mu, unfold, model_pred=model, reco_signal=measured, title="")

### 2 - 5: Unfolded x-sec uncert. budget

In [ ]:
sr_sig_covs_dict = {
    #"GENIE": genie_cov["cov_ms_ms"].item()["cov"],
    "Flux": unfold['CovRotation'] @ flux_cov["cov_ms_ms"].item()["cov"] @ unfold['CovRotation'].T,
    "MC Stat.": unfold['CovRotation'] @ stat_cov["cov_ms_ms"].item()["cov"] @ unfold['CovRotation'].T,
}
sr_bkg_covs_dict = {
    #"GENIE": genie_cov["cov_bs_bs"].item()["cov"],
    "Flux": unfold['CovRotation'] @ flux_cov["cov_bs_bs"].item()["cov"] @ unfold['CovRotation'].T,
    "MC Stat.": unfold['CovRotation'] @ stat_cov["cov_bs_bs"].item()["cov"] @ unfold['CovRotation'].T,
}

plot_combined_uncertainty_budget(varcfg_p_mu, unfold["unfold"]/2, unfold['unfold']/2, sr_sig_covs_dict, sr_bkg_covs_dict, title="SR Uncertainty Budget")

### 2 - 3: CCBC

Merge SB asimov data's stat unc to total cov in SB

In [ ]:
genie_cov_cc = genie_cov["cov_mc_mc"].item()["cov"] + genie_cov["cov_bc_bc"].item()["cov"] + genie_cov["cov_mc_bc"].item()["cov"] + genie_cov["cov_bc_mc"].item()["cov"]
flux_cov_cc = flux_cov["cov_mc_mc"].item()["cov"] + flux_cov["cov_bc_bc"].item()["cov"] + flux_cov["cov_mc_bc"].item()["cov"] + flux_cov["cov_bc_mc"].item()["cov"]
stat_cov_cc = stat_cov["cov_mc_mc"].item()["cov"] + stat_cov["cov_bc_bc"].item()["cov"] + stat_cov["cov_mc_bc"].item()["cov"] + stat_cov["cov_bc_mc"].item()["cov"]
cov_cc = flux_cov_cc + stat_cov_cc #+ genie_cov_cc
tilde_cov_cc = cov_cc + cov_asimov_data_sb

Constrained bs and cov_bs_bs

In [ ]:
## since d_C - n_c^CV is zero, bs_const = genie_cov["bs"]
genie_cov_bs_nc = (genie_cov["cov_bs_bc"].item()["cov"] + genie_cov["cov_bs_mc"].item()["cov"])
flux_cov_bs_nc = (flux_cov["cov_bs_bc"].item()["cov"] + flux_cov["cov_bs_mc"].item()["cov"])
stat_cov_bs_nc = (stat_cov["cov_bs_bc"].item()["cov"] + stat_cov["cov_bs_mc"].item()["cov"])
cov_bs_nc = flux_cov_bs_nc + stat_cov_bs_nc #+ genie_cov_bs_nc
cov_bs_nc_T = cov_bs_nc.T
tilde_cov_cc_inv = collect_inv_cov(tilde_cov_cc)
bs_const = genie_cov["bs"] + cov_bs_nc @ tilde_cov_cc_inv @ (asimov_data_sb - genie_cov['bc'] - genie_cov['mc'])

In [ ]:
#cov_bs_bs_const = flux_cov["cov_bs_bs"].item()["cov"] + stat_cov["cov_bs_bs"].item()["cov"] + genie_cov["cov_bs_bs"].item()["cov"] - cov_bs_nc @ tilde_cov_cc_inv @ cov_bs_nc_T
cov_bs_bs_const = flux_cov["cov_bs_bs"].item()["cov"] + stat_cov["cov_bs_bs"].item()["cov"] - cov_bs_nc @ tilde_cov_cc_inv @ cov_bs_nc_T

Unconst vs. Const for Asimov - background: merge asimov stat unc to bkg cov

In [ ]:
n_bins = len(varcfg_p_mu.bins) - 1
cov_zero = np.zeros((n_bins, n_bins))
arr_zero = np.zeros(n_bins)

asimov_m_bkg_sr = asimov_data_sr - genie_cov["bs"]
asimov_m_bkg_sr_const = asimov_data_sr - bs_const

#cov_asimov_m_bkg_sr = flux_cov["cov_bs_bs"].item()["cov"] + stat_cov["cov_bs_bs"].item()["cov"] + genie_cov["cov_bs_bs"].item()["cov"] + cov_asimov_data_sr
cov_asimov_m_bkg_sr = flux_cov["cov_bs_bs"].item()["cov"] + stat_cov["cov_bs_bs"].item()["cov"] + cov_asimov_data_sr
cov_asimov_m_bkg_sr_const = cov_bs_bs_const + cov_asimov_data_sr 

In [ ]:
plot_overlay_with_cov(
    data=asimov_m_bkg_sr,
    cov_data=cov_asimov_m_bkg_sr,
    signal=arr_zero,
    cov_signal=cov_zero,
    background=asimov_m_bkg_sr_const,
    cov_background= cov_asimov_m_bkg_sr_const,
    varcfg=varcfg_p_mu,
    title=f"SR: Asimov - Background (flux, GENIE, MC stat. unc.)",
    ylims=(-1e-41, 6e-41),
    bkg_label="Asimov - bkg",
    sig_label="Nothing",
    sig_p_bkg_label="",
    pred_unc_label="Post-CCBC Unc.",
    data_label="Pre-CCBC Unc.",
    y_label_top=r"Flux avg. $\sigma$ [cm$^2/\rm{Ar}$]",
)
